In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *

source_path = "/Volumes/workspace/madreporting/volumns/customerinjection/"
files = dbutils.fs.ls(source_path)

c_schema = StructType([
    StructField('customer_id',IntegerType(),False),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType()),
    StructField('load_date',StringType()),
])

a_schema = StructType([
    StructField('account_id',IntegerType()),
    StructField('customer_id',IntegerType(),False),
    StructField('account_type',StringType()),
    StructField('branch_city',StringType()),
    StructField('status',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType())
])

t_schema = StructType(
    [
        StructField('transaction_id',IntegerType()),
        StructField('account_id',IntegerType()),
        StructField('transaction_type',StringType()),
        StructField('amount',IntegerType()),
        StructField('transaction_date',StringType())
    ]
)

for file in files:
    if file.name.startswith('customer'):
        print('customer file found!')
        customerInc = (spark.read.format('csv').option('header','true').schema(c_schema).load(file.path))
    elif file.name.startswith('account'):
        print('account file found!')
        accountInc = (spark.read.format('csv').option('header','true').schema(a_schema).load(file.path))
    elif file.name.startswith('transaction'):
        print('transaction file found!')
        transInc = (spark.read.format('csv').option('header','true').schema(t_schema).load(file.path))

# drop nulls
customerInc = customerInc.dropna(subset=('customer_id'))
accountInc = accountInc.dropna(subset=('customer_id'))
transInc = transInc.dropna(subset=('transaction_id'))

# remove duplicates l1
customerInc = customerInc.dropDuplicates()
accountInc = accountInc.dropDuplicates()
transInc = transInc.dropDuplicates()

# remove duplicates l1
customerInc= customerInc.withColumn('rn',f.row_number().over(Window.partitionBy('customer_id').orderBy(f.col('load_date').desc()))).filter(f.col('rn')==1).drop('rn','load_date')

# null handling  
customerInc = customerInc.fillna({
    'customer_name':'Unknown',
    'city':'Unknown',
    'customer_segment':'Unknown',
    'email':'Unknown',
    'phone':'Unknown'
})

accountInc = accountInc.fillna({
    'account_type':'Unknown',
    'branch_city':'Unknown',
    'status':'Unknown'
})

transInc = transInc.fillna({
    'transaction_type':'Unknown'
})

# standardization
phone_regex = r'[^0-9]'
customerInc = customerInc.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.lower(f.col('email'))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,'')).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

accountInc = accountInc.withColumn('account_type',f.initcap(f.trim(f.col('account_type')))).withColumn('branch_city',f.initcap(f.trim('branch_city'))).withColumn('status',f.initcap(f.trim('status'))).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

transInc = transInc.withColumn('transaction_type',f.initcap(f.trim(f.col('transaction_type')))).orderBy(f.col('account_id'))

# data validation customer_id,customer_name,city,customer_segment,email,phone
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation = (f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)
amount_validation = (f.col('amount') >= 0)

customerInc = customerInc.withColumn('data_validation',f.when((email_validation) & (phone_validation),'Pass').otherwise('Fail'))
accountInc = accountInc.withColumn('data_validation',f.when((f.col('status')== 'Active'),'Pass').otherwise('Fail'))
transInc = transInc.withColumn('data_validation',f.when((amount_validation),'Pass').otherwise('Fail'))

customerInc = customerInc.filter(f.col('data_validation')=='Pass')
accountInc = accountInc.filter(f.col('data_validation')=='Pass')
transInc = transInc.filter(f.col('data_validation')=='Pass')

# add columns to match the existing table schema
customerInc = customerInc.withColumn('start_date',f.current_date()).withColumn(' ',f.lit('9999-12-31').cast('string')).withColumn('is_current',f.lit('Y'))

# audit columns
customerInc = customerInc.withColumn('updated_customer_dt',f.lit('9999-12-31')).withColumn('created_customer_dt',f.current_date().cast("string")).drop('data_validation')
accountInc = accountInc.withColumn('updated_account_dt',f.lit('9999-12-31')).withColumn('created_account_dt',f.current_date().cast("string")).drop('data_validation','start_date','end_date','is_current')
transInc = transInc.withColumn('updated_transaction_dt',f.lit('9999-12-31')).withColumn('created_transaction_dt',f.current_date().cast("string")).drop('data_validation')

# access the current tables 
customers = spark.table('workspace.madreporting.customers')
accounts = spark.table('workspace.madreporting.accounts')
transactions = spark.table('workspace.madreporting.transactions')

# create temporary tables
customerInc.createOrReplaceTempView('customerInc')
accountInc.createOrReplaceTempView('accountInc')
transInc.createOrReplaceTempView('transInc')

# customerInc.printSchema()
# customers.printSchema()

# scd2 implemention
spark.sql("""
          merge into workspace.madreporting.customers AS target
          using customerInc as Source
          on target.customer_id = Source.customer_id
          when matched then 
          update set
          target.customer_name = source.customer_name,
          target.city = source.city,
          target.customer_segment = source.customer_segment,
          target.email = source.email,
          target.phone = source.phone,
          target.updated_customer_dt = current_date()
          when not matched then
          insert *
          """)

print('scd1 implemented') 

# customerInc.show()
# accountInc.show()
# transInc.show()
